# 01 · 复现与调整 top-PC → tail 图

从这里开始：**Run All 不重新拟合**，直接读取已有 CSV。默认 (p=500)，可改成 100。下面保留完整 Matplotlib 绘图代码，方便修改颜色、范围、尺度、MC 汇总方式。

实验使用 Van Hateren 的实测协方差谱与 disk teacher，**MC 是 Gaussian feature designs**，不是重新抽取自然图像。每个噪声水平的 alpha 由 DE 的 LOOCV 风险选择，然后固定用于全部 MC trials；这里没有逐 trial 执行 RidgeCV。

对于 Gaussian 输入，固定 C、h、S 和噪声即固定特征与标签的联合分布；相同 MC fits 被共享到所有旋转位置。因此曲线沿旋转严格平坦来自构造与配对，而不是各位置独立重复实验的巧合。


In [ ]:
from pathlib import Path
import sys, time, json, hashlib, os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/accentuationpredrmt-matplotlib')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp/accentuationpredrmt-xdg-cache')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from threadpoolctl import threadpool_limits

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "rmt_core").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository or its notebooks directory.")
sys.path.insert(0, str(ROOT))
# Avoid excessive BLAS threads on shared machines.
blas_limit = threadpool_limits(limits=2)
from scripts import validate_vanhateren_top_pc_tail_rotations as experiment
OUT = ROOT / "notebooks" / "outputs" / "top_pc_rotation"
OUT.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)


In [ ]:
P = 500                     # 100 or 500
CMAP = "viridis"             # e.g. "plasma", "cividis"
MARKER_EVERY = 8
R2_STAT = "median"           # "mean", "median", "q25", "q75"
summary_path = ROOT / "tables/vanhateren_top_pc_tail_rotation_summary.csv"
df = pd.read_csv(summary_path)
display(df[df.p == P].head())


## 原图的精确重绘

调用原脚本的绘图函数，输出写入 notebook 专用目录，不覆盖之前的图片。

In [ ]:
reference = OUT / f"top{P}_original_layout.png"
experiment.plot_dimension(df.to_dict("records"), P, reference)
display(Image(filename=str(reference)))


## 可编辑绘图

运行下面两格即可调整外观。返回 fig 和 axes，可以继续设置坐标范围或导出 PDF。

In [ ]:
def plot_experiment(frame, p=500, cmap="viridis", marker_every=8,
                    r2_stat="median", figsize=(15, 8)):
    """Editable plot: lines = leading DE; circles = MC summaries."""
    data = frame[frame.p == p].copy()
    ratios = sorted(data.noise_signal_ratio.unique())
    colors = plt.get_cmap(cmap)(np.linspace(.06, .92, len(ratios)))
    fig, axes = plt.subplots(2, 3, figsize=figsize, sharex=True)
    base = data[np.isclose(data.noise_signal_ratio, ratios[0])].sort_values("tail_fraction")
    axes[0, 0].plot(base.tail_fraction, base.trace_g, color=".2")
    axes[0, 0].set(yscale="log", ylabel=r"$\mathrm{Tr}(FF^\top)$",
                   title=f"Unwhitened top-{p} PCs → tail")
    rank_ax = axes[0, 0].twinx()
    rank_ax.plot(base.tail_fraction, base.effective_rank_g, "--", color=".6")
    rank_ax.set_ylabel("effective rank", color=".5")
    panels = [
        (axes[0, 1], "gen_error_normalized", r"$E_{gen}/S$", "log"),
        (axes[0, 2], "r2_gen", r"$R^2_{gen}$", "linear"),
        (axes[1, 0], "acc_error_normalized", r"$E_{acc}/S$", "log"),
        (axes[1, 1], "r2_acc", r"$R^2_{acc}$", "symlog"),
        (axes[1, 2], "slope_acc", r"$\mathrm{slope}_{acc}$", "log"),
    ]
    for ratio, color in zip(ratios, colors):
        part = data[np.isclose(data.noise_signal_ratio, ratio)].sort_values("tail_fraction")
        label = rf"$\sigma={part.sigma.iloc[0]:.3g}$ ($\sigma^2/S={ratio:g}$)"
        for ax, metric, ylabel, scale in panels:
            ax.plot(part.tail_fraction, part[f"de_{metric}"], color=color, label=label)
            stat = r2_stat if metric == "r2_acc" else "mean"
            column = f"mc_{metric}_{stat}"
            if column in part:
                subset = part.iloc[::marker_every]
                ax.plot(subset.tail_fraction, subset[column], "o",
                        color=color, ms=4, mec=".2", mew=.3)
            ax.set_ylabel(ylabel)
            ax.set_title(ylabel)
            if scale == "symlog":
                ax.set_yscale(scale, linthresh=1)
            else:
                ax.set_yscale(scale)
    axes[1, 1].axhline(0, ls="--", color=".6", lw=.8)
    axes[0, 1].legend(fontsize=8, loc="best")
    for ax in axes.flat:
        ax.set_xscale("symlog", linthresh=1e-6, linscale=.6)
        ax.grid(alpha=.18)
    for ax in axes[1]:
        ax.set_xlabel(r"tail loading $t=\sin^2\theta$")
    fig.suptitle(f"Van Hateren disk teacher: p={p}; DE lines, MC circles")
    fig.tight_layout()
    return fig, axes


In [ ]:
fig, axes = plot_experiment(df, P, CMAP, MARKER_EVERY, R2_STAT)
# Example: axes[1, 1].set_ylim(-1e4, 1.1)
# Example: axes[1, 0].set_yscale("linear")
fig.savefig(OUT / f"top{P}_editable.png", dpi=180, bbox_inches="tight")
plt.show()


## 检查端点、alpha 与不变量

E_acc/S=(1-N/D)^2；R²_acc=1-(D/N-1)^2。两者分母不同，所以 E_acc/S 接近 1 时，R²_acc 可以极负。这不是相关系数平方。


In [ ]:
selected = df[df.p == P]
columns = ["noise_signal_ratio", "sigma", "tail_fraction", "de_alpha_cv",
           "de_r2_gen", "de_r2_acc", "mc_r2_acc_median", "de_slope_acc"]
display(selected[selected.tail_fraction.isin([0, 1])][columns])
display(selected.groupby("noise_signal_ratio")[
    ["de_gen_error_normalized", "mc_gen_error_normalized_mean",
     "de_r2_gen", "mc_r2_gen_mean", "de_alpha_cv"]
].agg(lambda x: x.max() - x.min()))


## 可选：原始 MC 分布

CSV 足够重新绘图。需要检查离群值、均值与中位数时才读取 NPZ 中的一组数组。原始缓存约 132 MB；若缓存不可用，可用 notebook 02 重新计算。


In [ ]:
raw_path = ROOT / "tables/vanhateren_top_pc_tail_rotation_cases.npz"
if raw_path.exists():
    with np.load(raw_path) as raw:
        trials = raw[f"p{P}_mc_r2_acc"]  # angle × noise × trial
        t = raw["tail_fraction"]
        ratios = raw["ratios"]
    ratio_index = 0
    angle_index = -1
    values = trials[angle_index, ratio_index]
    display(pd.Series(values).describe(percentiles=[.05, .25, .5, .75, .95]))
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(values, bins=40)
    ax.set(xlabel=r"$R^2_{acc}$", ylabel="trials",
           title=f"p={P}, t={t[angle_index]:g}, noise/signal={ratios[ratio_index]:g}")
    plt.show()
else:
    print("Raw cache unavailable; summary plots above remain usable.")
